# Deeper investigation — Plans and repeated work

**Worked solution** · [All exercises](../../index.html) · [Setup](../../README.md)

## What you’ll learn

- Read formatted plans to locate the work introduced by joins and aggregation.
- Request caching, materialise the report, inspect the cached plan, and release the cached data.

Optional. Complete [Exercise 5](../05-save-report.ipynb) and its **Save and finish** cell first. This investigation uses the same saved work; it does not replace your core pipeline.

Completed answers use a separate solution workspace and do not replace participant work.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](../00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../../docs/RECOVERY.md).

In [1]:
import sys
from pathlib import Path

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'lab_support/runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

from lab_support import checks as check
from lab_support.arrival_files import publish_arrival
from lab_support.checks import todo
from lab_support.runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path
from lab_support.workspace import Workspace

workspace = Workspace(solutions=True)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales, enrich_sales, category_totals = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales', 'enrich_sales', 'category_totals')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
enriched = enrich_sales(accepted, products)
report = category_totals(enriched)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 10:50:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0; inputs: data; notebook ready


---
<a id="extension-plans"></a>
## Your task

Inspect formatted plans for a projection, the enriched sales and the category report. Where does the work expand?

Make `cached_report = report.cache()`, request its rows, then inspect its plan. Finally release it with `unpersist()`. Compare values before and after. Do not use timings from this tiny fixture as evidence of a speedup.

In [2]:
raw.select("product_id").explain(mode="formatted")
enriched.explain(mode="formatted")
report.explain(mode="formatted")
cached_report = report.cache()
try:
    cached_report.show()
    cached_report.explain(mode="formatted")
    check.same_report(report, cached_report)
finally:
    cached_report.unpersist()

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [product_id#1]
Batched: true
Location: InMemoryFileIndex [file:<lab-root>/data/sales.parquet]
ReadSchema: struct<product_id:string>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [product_id#1]




== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- BroadcastHashJoin LeftOuter BuildRight (8)
      :- Project (3)
      :  +- Filter (2)
      :     +- Scan parquet  (1)
      +- BroadcastExchange (7)
         +- Project (6)
            +- Filter (5)
               +- Scan parquet  (4)


(1) Scan parquet 
Output [4]: [sale_id#0, product_id#1, amount_raw#2, sold_at_raw#3]
Batched: true
Location: InMemoryFileIndex [file:<lab-root>/data/sales.parquet]
ReadSchema: struct<sale_id:string,product_id:string,amount_raw:string,sold_at_raw:string>

(2) Filter
Input [4]: [sale_id#0, product_id#1, amount_raw#2, sold_at_raw#3]
Condition : CASE WHEN (isnull(upper(trim(product_id#1, None))) OR (upper(trim(product_id#1, None)) = )) THEN false WHEN isnull(try_cast(amount_raw#2 as decimal(12,2))) THEN false WHEN isnull(gettimestamp(sold_at_raw#3, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_timestamp, Some(UTC), false)) THEN false ELSE true END

(3) Project
Output [4]: [sale_id#0, upper(trim(

+--------+-----+-----+
|category|sales|total|
+--------+-----+-----+
|   books|    3|50.00|
|unmapped|    1|10.00|
|   games|    1|40.00|
+--------+-----+-----+

== Physical Plan ==
AdaptiveSparkPlan (13)
+- HashAggregate (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Project (9)
            +- BroadcastHashJoin LeftOuter BuildRight (8)
               :- Project (3)
               :  +- Filter (2)
               :     +- Scan parquet  (1)
               +- BroadcastExchange (7)
                  +- Project (6)
                     +- Filter (5)
                        +- Scan parquet  (4)


(1) Scan parquet 
Output [3]: [product_id#1, amount_raw#2, sold_at_raw#3]
Batched: true
Location: InMemoryFileIndex [file:<lab-root>/data/sales.parquet]
ReadSchema: struct<product_id:string,amount_raw:string,sold_at_raw:string>

(2) Filter
Input [3]: [product_id#1, amount_raw#2, sold_at_raw#3]
Condition : CASE WHEN (isnull(upper(trim(product_id#1, None))) OR (upper(trim(product_id#

In [3]:
assert not report.is_cached, "Release the cached report before continuing."

<details><summary>Hint</summary>

`cache()` requests persistence. An action materialises data; the request alone does not. Use `try/finally` to release persistence even if inspection fails.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [4]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Session stopped; exercise files are under runs/run-4c287fb99c


Return to [all exercises](../../index.html).